# Day 6
## JSON Web Token (JWT) Authentication
A JSON Web Token is a token __format__ containing user information.

A JWT may display as:
```bash
eyJhbGciOiJIUzI1NiIs...
```

But contain:
* Header - Metadata about the token and signing algorithm
* Payload - Claims about the user/entity
* Signature - verifies integrity and authenicity

A payload example:
```json
{
    "sub"   : "john"
    "exp"   : 17300000000
}
```

Common claims (key/value pairs) include:
| Claim     | Meaning           |
| :--       | :--               |
| sub       | subject (user id) |
| exp       | expiration        |
| iat       | issued at         |
| iss       | issuer            |
| aud       | audience          |


## Access Tokens
Access token provide authorisation, not identification.<br>
i.e. access token provides access to a resource whilst the JWT provides the identification.

They are generally short lived (5-30 minutes) and used on every API request.


## Refresh Tokens
Refresh tokens are long-lived credentials (days-months), used only to obtain new Access Tokens when they expire.

Whilst Access Tokens carry user claims and are frequently transmitted, Refresh Tokens are stored securely and rarely used, reducing exposure risk.


## [python-jose](https://pypi.org/project/python-jose/)
JavaScript Object Signing and Encryption (JOSE) is used for creating, verifying, and decoding JWTs in authentication systems.



## `current_user` Dependency
Example:<br>
A landing page requires that a user is currently logged in.<br>
The page __depends__ on the current_user.
```py
from fastapi import Depends
from jose import jwt

def get_current_user(           # 2.get_current_user()
    token: str = Depends(oauth2_scheme)
):
    payload = jwt.decode(       # 3.Decode
         token
        ,SECRET_KEY
        ,algorithms=["HS256"]
    )
    username = payload.get("sub")
    return {
        "username": username    # 4.Return user
    }

@router.get("/me)               # 1.Request   
def read_me(
    current_user = Depends (
        get_current_user        # 2.get_current_user()
    )
):
    return current_user         # 6.Route Runs
```

The workflow:
```bash
1.Request
    ↓
2.get_current_user()
    ↓
3.Decode JWT
    ↓
4.Find User
    ↓
5.Return User
    ↓
6.Route Runs
```

## OAuth2PasswordBearer
```py
from fastapi.security import OAuth2PasswordBearer
```

OAuth2PasswordBearer is used to implement OAuth Password Flow authentication.<br>
It does not verify toekns by itself but declares a dependency that extracts the token from incoming requests.

`OAuth2PasswordBearer` is used with JWT tokens to secure API endpoints in FastAPI.<br>
This allows FastAPI applications to implement stateless, scalable authentication without server-dide session storage, making it suitable for modern APIs.


## Redis Token Storage
Remote Dictionary Server (Redis) is an in-memory key–value database, used as a distributed cache and message broker, with optional durability.<br>
Because it holds all data in memory and because of its design, Redis offers low-latency reads and writes, making it particularly suitable for use cases that require a cache.



JWT is usually stateless - the server stores nothing.

> Stateless tokens carry all session information within the token itself, while stateful tokens rely on server-side storage to track session state.


Problem: How do users log out?

Once JWT is issued: Until the token expires.

Redis solution: Store refresh tokens.

```py
redis.set(
    "refresh:1",
    refresh_token
)

redis.get(
    "refresh:1"
)
```


## Token Revocation
```py
redis.delete(
    f"refresh:{user_id}"
)

# Alternatively

redis.delete(token)
```

```bash
User logs out
      ↓
DEL token
      ↓
Immediately invalid
```

